In [ ]:
import os
os.environ['USE_PYGEOS'] = '0'
import sys
sys.path.insert(1, os.path.abspath(".."))
from qtree_index import nestedgrid, mortoncurve, qtree

import numpy
import pandas
from datetime import datetime, timedelta
import pytz
import geopandas
import shapely
from shapely import wkb, wkt
import pyproj

import subprocess
import shutil
from glob import glob
import json

from functools import partial
from multiprocessing import Pool
from random import shuffle

import pyrosm
"""
pip install pyogrio
Important: edit the file: gdal_data/osmconf.ini in the pyogrio folder, e.g:
/data/mfreitag/software/my_311/lib/python3.11/site-packages/pyogrio/gdal_data/osmconf.ini

# uncomment to specify the the format for the all_tags/other_tags field should be JSON
# instead of the default HSTORE formatting.
# Valid values for tags_format are "hstore" and "json"
tags_format=json

Set these common attributes in 5 places (for 'points', 'lines', 'multilinestrings', 'multipolygons', 'other_relations')
osm_version=yes
osm_timestamp=yes
# uncomment to avoid creation of "other_tags" field
other_tags=no
# uncomment to create "all_tags" field. "all_tags" and "other_tags" are exclusive
all_tags=yes
"""
import pyogrio

import matplotlib.pyplot as plt

# OSMIUM: Split from Open Street Map (osm) planet file

In [ ]:
# Define the partitioning grid
grid = nestedgrid.World()
morton = mortoncurve.Morton(grid)

In [ ]:
# q_key = '0q0'
# buffer = 1
# bounds_buffered = shapely.box(*morton.base4_to_box(q_key).buffer(buffer).bounds).intersection(shapely.box(*grid.valid_bounds)).bounds

# # osmium extract
# minx, miny, maxx, maxy = bounds_buffered
# #minx, miny, maxx, maxy = -73.9,41.1,-73.8,41.2

# pbf_file = '/data/vector/osm/planet-latest.osm.pbf'
# out_file = f'/data/vector/osm/extract/{q_key}.pbf'

# command = f'osmium extract -s smart -S types=any -b {minx},{miny},{maxx},{maxy} {pbf_file} -o {out_file} '
# subprocess.run(command, shell=True, check=True)

In [ ]:
def buffered_bounds(q_key, buffer):
    return shapely.box(*morton.base4_to_box(q_key).buffer(buffer).bounds).intersection(shapely.box(*grid.valid_bounds)).bounds

def get_filepath(q_key):
    return f'/data/vector/osm/extract/{q_key}.pbf'

def extract_key(filepath):
    return os.path.basename(filepath).split('.pbf')[0]
    
def descend_tree_split_pbf(in_key, buffer, max_file_size=1e9, max_level=8):
    """Cut the osm data into 4 quadrants."""
    
    if in_key=='0q':
        in_file = f'/data/vector/osm/planet-latest.osm.pbf'
        # First level does not have valid keys '0q2' and '0q3'
        out_keys = [in_key+c for c in '01']
    else:
        in_file = f'/data/vector/osm/extract/{in_key}.pbf'
        # All four quadtree children
        out_keys = [in_key+c for c in '0123']

    in_level = len(in_key)-2

    if os.stat(in_file).st_size>max_file_size and (
        not all([os.path.exists(get_filepath(out_key)) for out_key in out_keys]) and
        in_level<max_level
    ):
        # Write osmium config file to disk
        config_file = f'/data/vector/osm/extract/{in_key}.json'
        extracts = {
            "extracts": [{
                "output": get_filepath(out_key),
                "output_format": "pbf",
                "bbox": list(buffered_bounds(out_key, buffer))
            } for out_key in out_keys]
        }
        with open(config_file, 'w') as f:
            json.dump(extracts, f)
        
        # osmium extract
        command = f'osmium extract -s smart -S types=any -v -c {config_file} {in_file}'
        print(command)
        
        subprocess.run(command, shell=True, check=True)
    
    return [out_key for out_key in out_keys if os.path.exists(get_filepath(out_key))]

In [ ]:
# Cut the osm data into 4 quadrants.
in_keys = ['0q']
max_level = 8
buffer = 0.1
max_file_size = 5e8 #1e9
while len(in_keys)>0:
    in_key = in_keys.pop()
    print('in_key ', in_key)
    in_keys += descend_tree_split_pbf(in_key, buffer, max_file_size=max_file_size, max_level=max_level)
    print('in_keys', in_keys)

In [ ]:
def descend_tree_copy_leafs(in_key, max_level=8):
    """Copy the files we want to work with (leaf nodes) to separate folder."""
    
    if in_key=='0q':
        in_file = f'/data/vector/osm/planet-latest.osm.pbf'
        # First level does not have valid keys '0q2' and '0q3'
        out_keys = [in_key+c for c in '01']
    else:
        in_file = f'/data/vector/osm/extract/{in_key}.pbf'
        # All four quadtree children
        out_keys = [in_key+c for c in '0123']

    in_level = len(in_key)-2

    if not any([os.path.exists(get_filepath(out_key)) for out_key in out_keys]) or in_level==max_level:
        # Arrived at a leaf node
        src = f'/data/vector/osm/extract/{in_key}.pbf'
        dst = f'/data/vector/osm/split/{in_key}.pbf'
        if not os.path.exists(dst):
            shutil.copyfile(src, dst)

        if round(os.stat(f'/data/vector/osm/extract/{in_key}.pbf').st_size/1e8)/10>2:
            print(round(os.stat(f'/data/vector/osm/extract/{in_key}.pbf').st_size/1e8)/10, src, '->', dst, '!!!!!!!!!!!!!!!!!!!!!!')
        else:
            print(round(os.stat(f'/data/vector/osm/extract/{in_key}.pbf').st_size/1e8)/10, src, '->', dst)
        
        return []
    else:
        out_keys = [out_key for out_key in out_keys if os.path.exists(get_filepath(out_key))]
        if in_key=='0q':
            assert len(out_keys)==2
        else:
            assert len(out_keys)==4
        return out_keys

In [ ]:
# Copy the files we want to work with (leaf nodes) to separate folder.

# Lets see if level 7 is deep enough (level 8 seems to improve the file sizes only marginally)
max_level=8
in_keys = ['0q']
while len(in_keys)>0:
    in_key = in_keys.pop()
    in_keys += descend_tree_copy_leafs(in_key, max_level=max_level)

In [ ]:
# Plot spatial extent and sizes of split files
split_folder = '/data/vector/osm/split'
pbf_files = sorted(glob(os.path.join(split_folder, '*.pbf')))
q_keys = [extract_key(f) for f in pbf_files]
gdf_sizes = pandas.DataFrame({'q_key': q_keys, 'filepath': pbf_files})
gdf_sizes['geometry'] = gdf_sizes['q_key'].apply(lambda x: morton.base4_to_box(x))
gdf_sizes = geopandas.GeoDataFrame(gdf_sizes)
gdf_sizes['size (GB)'] = gdf_sizes['filepath'].apply(lambda x: os.stat(x).st_size/1e9)
gdf_sizes.plot(gdf_sizes['size (GB)'], cmap='coolwarm', edgecolor='k', lw=.2, legend=True, figsize=(12, 5))
plt.title('split .pbf file size (GB)')
plt.show()

# pyogrio: Read .pbf and combine different kinds of geometries

In [ ]:
def _json_loads(series):
    """Convert json string to dictionary."""
    series = series.fillna('{}')
    series = series.apply(lambda x: json.loads(x, strict=False))
    
    return series

def _read_layer_pyogrio(src, layer, bbox, osmconf_error, filter_on_partition_key):
    """Extract some columns from all_tags and move the remaining ones to other_tags.

    Valid pyogrio layers: 'points', 'lines', 'multilinestrings', 'multipolygons', 'other_relations'
    """
    gdf = geopandas.read_file(src, engine="pyogrio", use_arrow=True, force_2d=True, tags_format='json', layer=layer)
    if not (set(['osm_version', 'osm_timestamp', 'all_tags']) <= set(gdf.columns)):
        raise Exception(osmconf_error)
        
    if filter_on_partition_key:
        gdf = gdf[gdf.intersects(bbox)].reset_index(drop=True)

    gdf['geometry'] = gdf['geometry'].make_valid()
        
    # Convert json string to dictionary.
    gdf['all_tags'] = _json_loads(gdf['all_tags'])

    # Since we combine different dataframes later, remember the layer here
    gdf['pyogrio_layer'] = layer

    # # Drop all-nan columns
    # gdf = gdf.dropna(axis=1, how='all')

    # Rename the id columns to reflect where they come from (mixed they woulud not be unique)
    if layer=='points':
        gdf = gdf.rename(columns={'osm_id': 'osm_node_id'})
    elif layer=='lines':
        gdf = gdf.rename(columns={'osm_id': 'osm_relation_id'})
    elif layer=='multilinestrings':
        gdf = gdf.rename(columns={'osm_id': 'osm_relation_id'})
    elif layer=='multipolygons':
        gdf = gdf.rename(columns={'osm_id': 'osm_relation_id', 'osm_way_id': 'osm_way_id'})
    elif layer=='other_relations':
        gdf = gdf.rename(columns={'osm_id': 'osm_relation_id'})

    return gdf

def _split_tags(tags_dict, cols):
    """Split all_tags json dictionaries into those we want to extract as columns and those we want to keep as json."""
    # Important, do not explode these tags, since these columns are set by us
    EXCLUDE_COLS = [
        'osm_node_id',
        'osm_relation_id',
        'osm_timestamp',
        'osm_version',
        'osm_way_id',
        'pyogrio_layer',
        'all_tags',
        'other_tags',
        'geometry',
    ]
    explode_cols = sorted(set(cols) - set(EXCLUDE_COLS))
    explode_tags = {x: tags_dict[x] for x in tags_dict if x in explode_cols}
    other_tags = {x: tags_dict[x] for x in tags_dict if x not in explode_cols}
    
    return explode_tags, other_tags

def _tags_to_columns(gdf, cols):
    """Extract some columns from all_tags and move the remaining ones to other_tags."""
    if len(gdf)>0:
        # Split all_tags json dictionaries into those we want to extract as columns and those we want to keep as json.
        gdf[['explode_tags', 'other_tags']] = gdf['all_tags'].apply(lambda x: _split_tags(x, cols)).to_list()
        
        # Convert explode_tags to columns
        gdf_tmp = pandas.DataFrame(gdf['explode_tags'].to_list())
        other_cols = sorted(set(gdf.columns) - set(gdf_tmp.columns) - set(['all_tags', 'explode_tags']))
        gdf = pandas.concat([gdf_tmp, gdf[other_cols]], axis=1)
        
        # Dumping other_tags to json string so that the column can be saved efficiently in parquet
        gdf['other_tags'] = gdf['other_tags'].apply(lambda x: json.dumps(x))
        
    return geopandas.GeoDataFrame(gdf)

def _prepend(x, letter):
    """Nan-tolerant string prepend"""
    try:
        return letter + x
    except TypeError:
        return None
    
def read_file_pyogrio(src, filter_on_partition_key=True):
    """
    Read split osm .pbf file and filter based on partition key.
    """
    if filter_on_partition_key:
        # Filter by base4 key.
        src_key = extract_key(src)
        bbox = morton.base4_to_box(src_key)
    else:
        bbox = None

    osmconf_error = """
        Python environmen ini file osmconf.ini, https://svn.osgeo.org/gdal/trunk/gdal/data/osmconf.ini, not edited correctly.
        
        After pip install pyogrio, edit the file: gdal_data/osmconf.ini in the venv pyogrio folder, e.g:
        /data/software/my_311/lib/python3.11/site-packages/pyogrio/gdal_data/osmconf.ini
        
        Set tags_format in one place:
        tags_format=json
        
        Set these common attributes in 5 places (for 'points', 'lines', 'multilinestrings', 'multipolygons', 'other_relations')
        osm_version=yes
        osm_timestamp=yes
        other_tags=no
        all_tags=yes
    """

    gdf_points = _read_layer_pyogrio(src, 'points', bbox, osmconf_error, filter_on_partition_key)
    gdf_lines = _read_layer_pyogrio(src, 'lines', bbox, osmconf_error, filter_on_partition_key)
    gdf_multilinestrings = _read_layer_pyogrio(src, 'multilinestrings', bbox, osmconf_error, filter_on_partition_key)
    gdf_multipolygons = _read_layer_pyogrio(src, 'multipolygons', bbox, osmconf_error, filter_on_partition_key)
    gdf_other_relations = _read_layer_pyogrio(src, 'other_relations', bbox, osmconf_error, filter_on_partition_key)
    
    # Union of all columns returned plus a set of minimum custom columns
    minimum_cols = [
        'name',
        'name:en',
        'aerialway',
        'aeroway',
        'amenity',
        'boundary',
        'building',
        'craft',
        'emergency',
        'geological',
        'highway',
        'historic',
        'landuse',
        'leisure',
        'natural',
        'office',
        'place',
        'power',
        'public_transport',
        'railway',
        'route',
        'shop',
        'tourism',
        'waterway',
    ]
    cols = sorted(
        set(gdf_points.columns) | 
        set(gdf_lines.columns) | 
        set(gdf_multilinestrings.columns) | 
        set(gdf_multipolygons.columns) | 
        set(gdf_other_relations.columns) |
        set(minimum_cols)
    )

    gdf_points = _tags_to_columns(gdf_points, cols)
    gdf_lines = _tags_to_columns(gdf_lines, cols)
    gdf_multilinestrings = _tags_to_columns(gdf_multilinestrings, cols)
    gdf_multipolygons = _tags_to_columns(gdf_multipolygons, cols)
    gdf_other_relations = _tags_to_columns(gdf_other_relations, cols)

    # Concat the five types of DataFrames
    gdf = pandas.concat([
        gdf_points,
        gdf_lines,
        gdf_multilinestrings,
        gdf_multipolygons,
        gdf_other_relations,
    ]).reset_index(drop=True)

    # Debug: In some cases z-order has both int, nan, and string values, so unify float
    if 'z_order' in gdf:
        gdf['z_order'] = gdf['z_order'].astype(float)

    # Create a unique id "osm_id" for each row from osm_id (os_node_id) and osm_way_id
    if 'osm_node_id' not in gdf:
        gdf['osm_node_id'] = numpy.nan
    if 'osm_relation_id' not in gdf:
        gdf['osm_relation_id'] = numpy.nan
    if 'osm_way_id' not in gdf:
        gdf['osm_way_id'] = numpy.nan
    gdf['osm_id'] = gdf['osm_node_id'].apply(lambda x: _prepend(x, 'n')) # n for osm_node_id
    gdf['osm_id'] = gdf['osm_id'].fillna(gdf['osm_relation_id'].apply(lambda x:_prepend(x, 'r'))) # w for osm_way_id)
    gdf['osm_id'] = gdf['osm_id'].fillna(gdf['osm_way_id'].apply(lambda x:_prepend(x, 'w'))) # w for osm_way_id)

    return geopandas.GeoDataFrame(gdf[sorted(gdf.columns)])

## Debug here if above failed for a src1_key

In [ ]:
# %%time
# # Debug

# # Testing on some of the largest files
# src1_key = '0q12022232'
# src1 = get_filepath(src1_key)

# gdf1 = read_file_pyogrio(src1)
# gdf1.tail(2)

# s = gdf1['geometry'].apply(lambda x: isinstance(x, shapely.geometry.base.BaseGeometry))
# gdf_bad = gdf1[~s]
# gdf_good = gdf1[s]
# print(len(gdf1), len(gdf_good), len(gdf_bad))

# dst1 = os.path.join(
#     '/data/vector/osm/intermediate_parquet',
#     os.path.splitext(os.path.basename(src1))[0] + '.parquet'
# )
# gdf1.to_parquet(
#     path=dst1,
#     engine='pyarrow',
#     compression='snappy',
# )

# gdf_read = geopandas.read_parquet(dst1)
# gdf_read

## Parallel processing from pbf to (intermediate) parquet

In [ ]:
def _dst(src, out_directory):
    out_filename = os.path.splitext(os.path.basename(src))[0] + '.parquet'
    dst = os.path.join(out_directory, out_filename)
    return dst

def process_pbf_to_parquet(src, out_directory):
    try:
        gdf = read_file_pyogrio(src)
        if len(gdf)>0:
            gdf.to_parquet(
                path=_dst(src, out_directory),
                engine='pyarrow',
                compression='snappy',
            )
    except:
        print('FAILED:', src)
        
    return

In [ ]:
intermediate_directory='/data/vector/osm/intermediate_parquet'

lst_src = glob('/data/vector/osm/split/*.pbf')
lst_src = [src for src in lst_src if not os.path.exists(_dst(src, intermediate_directory))]
shuffle(lst_src)
print(len(lst_src))
lst_src

In [ ]:
# Do the heavy lifting in parallel
n_workers = 6
process_pbf_to_parquet_part = partial(process_pbf_to_parquet, out_directory=intermediate_directory)
with Pool(n_workers) as p:
    p.map(process_pbf_to_parquet_part, lst_src)

# Split into 11 different osm collections

In [ ]:
def split_collections(src, osm_keys, collections_directory):
    """Split osm parquet file into separate osm collections."""
    gdf = geopandas.read_parquet(src)
    basename = os.path.basename(src)
    for key in osm_keys: #['road', 'poi', 'natural']:
        cols = list(set(gdf.columns) & set(osm_keys[key]))
        if len(cols)>0:
            print(key, cols)
            gdf_split = gdf[gdf[cols].notnull().any(axis=1)].reset_index(drop=True)
            collection_directory = os.path.join(collections_directory, key)
            if not os.path.exists(collection_directory): os.makedirs(collection_directory)
            if len(gdf_split)>0:
                gdf_split.to_parquet(
                    path=os.path.join(collection_directory, basename),
                    engine='pyarrow',
                    compression='snappy',
                )

In [ ]:
# Dictionary of osm collection keys and non-nan osm column names as values
osm_keys = {}
osm_keys['boundary'] = ['boundary']
osm_keys['building'] = ['building']
osm_keys['road'] = ['highway'] # We use "raod" collection, for non-nan "highway" keys.
osm_keys['landuse'] = ['landuse']
osm_keys['power'] = ['power']
osm_keys['railway'] = ['railway']
osm_keys['waterway'] = ['waterway']
osm_keys['natural'] = ['natural', 'geological'] # non-nan "geological" and "natural" keys included in "natural" collection.
osm_keys['poi'] = ['amenity', 'shop', 'tourism', 'craft', 'emergency', 'historic', 'office', 'sport', 'leisure', 'place', 'aeroway']

osm_collections = sorted(osm_keys.keys())
print('osm_collections', osm_collections)
print('osm_keys:')
osm_keys

In [ ]:
collections_directory = '/data/vector/osm/collections'
intermediate_directory='/data/vector/osm/intermediate_parquet'
lst_files = glob(os.path.join(intermediate_directory, '*.parquet'))
len(lst_files)

In [ ]:
for src in lst_files:
    print(src)
    split_collections(src, osm_keys, collections_directory)

In [ ]:
%%time
gdf = geopandas.read_parquet('/data/vector/osm/collections/power/0q01210.parquet')
len(gdf)